# Expected Goals (xG) Model — StatsBomb Open Data

**Author:** Anil Turan  
**Goal:** Build a logistic regression xG model using StatsBomb open data.

### Pipeline
1. Load StatsBomb shot data
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Logistic Regression xG Model
5. Evaluation & Calibration
6. Visualisation — xG Shot Map

## 0. Setup
```bash
pip install statsbombpy pandas numpy scikit-learn matplotlib seaborn
```

In [ ]:
import sys
!{sys.executable} -m pip install statsbombpy pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from statsbombpy import sb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42

## 1. Load Data

StatsBomb provides free open data for selected competitions.  
We'll use **La Liga 2015/16** (Messi's famous season — rich shot data).

In [ ]:
# Browse available free competitions
competitions = sb.competitions()
print(competitions[['competition_id', 'season_id', 'competition_name', 'season_name']].to_string())

In [ ]:
# La Liga 2015/16: competition_id=11, season_id=27
COMPETITION_ID = 11
SEASON_ID = 27

matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
print(f"Total matches: {len(matches)}")
matches.head(3)

In [ ]:
# Load all shots from all matches
all_shots = []

for match_id in matches['match_id']:
    events = sb.events(match_id=match_id)
    shots = events[events['type'] == 'Shot'].copy()
    all_shots.append(shots)

shots_df = pd.concat(all_shots, ignore_index=True)
print(f"Total shots loaded: {len(shots_df)}")
print(f"Columns: {shots_df.columns.tolist()}")

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic stats
print("=== Shot Outcome Distribution ===")
print(shots_df['shot_outcome'].value_counts())
print(f"\nGoal rate: {(shots_df['shot_outcome'] == 'Goal').mean():.3f}")

In [ ]:
# Extract x, y from location column
shots_df['x'] = shots_df['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else None)
shots_df['y'] = shots_df['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else None)
shots_df = shots_df.dropna(subset=['x', 'y'])

print(f"x range: {shots_df['x'].min():.1f} – {shots_df['x'].max():.1f}")
print(f"y range: {shots_df['y'].min():.1f} – {shots_df['y'].max():.1f}")

In [ ]:
# Shot map: all shots coloured by outcome
def draw_pitch(ax, pitch_color='white', line_color='black'):
    """Draw a StatsBomb-scaled football pitch (120x80)."""
    ax.set_facecolor(pitch_color)
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 80)
    ax.set_aspect('equal')
    ax.axis('off')

    # Pitch outline & centre line
    for shape in [
        patches.Rectangle((0, 0), 120, 80, fill=False, edgecolor=line_color, lw=2),
        patches.Rectangle((0, 22.3), 18, 35.4, fill=False, edgecolor=line_color, lw=1.5),   # left box
        patches.Rectangle((102, 22.3), 18, 35.4, fill=False, edgecolor=line_color, lw=1.5), # right box
        patches.Rectangle((0, 30), 6, 20, fill=False, edgecolor=line_color, lw=1),          # left 6yd
        patches.Rectangle((114, 30), 6, 20, fill=False, edgecolor=line_color, lw=1),        # right 6yd
    ]:
        ax.add_patch(shape)

    # Centre circle & spot
    ax.add_patch(plt.Circle((60, 40), 10, fill=False, edgecolor=line_color, lw=1.5))
    ax.plot([60, 60], [0, 80], color=line_color, lw=1.5)  # halfway line
    ax.plot(60, 40, 'o', color=line_color, ms=3)

fig, ax = plt.subplots(figsize=(14, 9))
draw_pitch(ax, pitch_color='#f5f5f5')

goals = shots_df[shots_df['shot_outcome'] == 'Goal']
non_goals = shots_df[shots_df['shot_outcome'] != 'Goal']

ax.scatter(non_goals['x'], non_goals['y'], c='steelblue', alpha=0.3, s=25, label='No Goal', zorder=3)
ax.scatter(goals['x'], goals['y'], c='crimson', alpha=0.8, s=60, label='Goal', zorder=4, marker='*')

ax.set_title('Shot Map — La Liga 2015/16', fontsize=16, fontweight='bold', pad=15)
ax.legend(loc='upper left', fontsize=12)
plt.tight_layout()
plt.savefig('shot_map.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Goal rate by shot body part
body_part_rates = (
    shots_df.groupby('shot_body_part')
    .apply(lambda g: (g['shot_outcome'] == 'Goal').mean())
    .reset_index(name='goal_rate')
    .sort_values('goal_rate', ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=body_part_rates, x='shot_body_part', y='goal_rate', palette='Blues_d', ax=ax)
ax.set_title('Goal Rate by Body Part', fontsize=14)
ax.set_ylabel('Goal Rate')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('goal_rate_body_part.png', dpi=150)
plt.show()

## 3. Feature Engineering

Classic xG features:
- **distance** — Euclidean distance from shot location to goal centre
- **angle** — angle subtended by the goal mouth from shot location
- **is_header** — shot taken with head
- **is_open_play** — open play vs set piece / counter
- **strong_foot** — proxy for dominant foot (Right foot as baseline)

In [ ]:
GOAL_X = 120.0
GOAL_Y_LEFT = 36.0
GOAL_Y_RIGHT = 44.0
GOAL_Y_CENTRE = 40.0

def compute_distance(x, y):
    return np.sqrt((GOAL_X - x)**2 + (GOAL_Y_CENTRE - y)**2)

def compute_angle(x, y):
    """Angle (radians) subtended by goal mouth at shot location."""
    a = np.sqrt((GOAL_X - x)**2 + (GOAL_Y_LEFT - y)**2)
    b = np.sqrt((GOAL_X - x)**2 + (GOAL_Y_RIGHT - y)**2)
    c = abs(GOAL_Y_RIGHT - GOAL_Y_LEFT)  # goal width = 8 yards
    cos_angle = (a**2 + b**2 - c**2) / (2 * a * b)
    cos_angle = np.clip(cos_angle, -1, 1)
    return np.arccos(cos_angle)

shots_df['distance'] = compute_distance(shots_df['x'], shots_df['y'])
shots_df['angle']    = compute_angle(shots_df['x'], shots_df['y'])
shots_df['is_header']    = (shots_df['shot_body_part'] == 'Head').astype(int)
shots_df['is_open_play'] = (shots_df['shot_type'] == 'Open Play').astype(int)
shots_df['goal']         = (shots_df['shot_outcome'] == 'Goal').astype(int)

# Distance bins for EDA
shots_df['dist_bin'] = pd.cut(shots_df['distance'], bins=[0,10,15,20,25,30,40,100])

print(shots_df[['distance', 'angle', 'is_header', 'is_open_play', 'goal']].describe())

In [ ]:
# Goal rate vs distance
dist_goal_rate = (
    shots_df.groupby('dist_bin')
    .agg(shots=('goal', 'count'), goals=('goal', 'sum'))
    .assign(goal_rate=lambda d: d['goals'] / d['shots'])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(len(dist_goal_rate)), dist_goal_rate['goal_rate'], color='steelblue', alpha=0.8)
ax.set_xticks(range(len(dist_goal_rate)))
ax.set_xticklabels([str(b) for b in dist_goal_rate['dist_bin']], rotation=30, ha='right')
ax.set_ylabel('Goal Rate')
ax.set_xlabel('Distance to Goal (m)')
ax.set_title('Goal Rate by Distance', fontsize=14)
plt.tight_layout()
plt.savefig('goal_rate_distance.png', dpi=150)
plt.show()

## 4. Logistic Regression xG Model

In [ ]:
FEATURES = ['distance', 'angle', 'is_header', 'is_open_play']
TARGET   = 'goal'

model_df = shots_df[FEATURES + [TARGET]].dropna()

X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train: {len(X_train)} shots | Test: {len(X_test)} shots")
print(f"Train goal rate: {y_train.mean():.3f} | Test goal rate: {y_test.mean():.3f}")

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=SEED)
model.fit(X_train, y_train)

# Print coefficients
coef_df = pd.DataFrame({'feature': FEATURES, 'coefficient': model.coef_[0]})
print("=== Model Coefficients ===")
print(coef_df.to_string(index=False))

## 5. Evaluation

In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]

auc   = roc_auc_score(y_test, y_pred_proba)
brier = brier_score_loss(y_test, y_pred_proba)

print(f"ROC-AUC   : {auc:.4f}")
print(f"Brier Score: {brier:.4f}  (lower = better, baseline ~{y_test.mean()*(1-y_test.mean()):.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- ROC Curve ---
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'Logistic Reg (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1], 'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# --- Calibration Curve ---
fraction_pos, mean_pred = calibration_curve(y_test, y_pred_proba, n_bins=10)
axes[1].plot(mean_pred, fraction_pos, 'o-', color='steelblue', lw=2, label='Model')
axes[1].plot([0,1],[0,1], 'k--', lw=1, label='Perfect calibration')
axes[1].set_xlabel('Mean Predicted xG')
axes[1].set_ylabel('Fraction of Goals')
axes[1].set_title('Calibration Curve')
axes[1].legend()

plt.suptitle('xG Model Evaluation', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. xG Shot Map — Predicted Values

In [ ]:
# Add xG predictions to full dataset
shots_df['xg'] = model.predict_proba(shots_df[FEATURES].dropna())[:, 1]

# Focus on attacking half (x > 60)
plot_shots = shots_df[shots_df['x'] > 60].copy()

fig, ax = plt.subplots(figsize=(12, 8))
draw_pitch(ax, pitch_color='#f0f0f0')

# Only show attacking half
ax.set_xlim(60, 120)
ax.set_ylim(0, 80)

sc = ax.scatter(
    plot_shots['x'], plot_shots['y'],
    c=plot_shots['xg'],
    cmap='RdYlGn',
    s=plot_shots['xg'] * 300 + 15,  # size proportional to xG
    alpha=0.7,
    edgecolors='grey',
    linewidths=0.3,
    zorder=3
)

cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('xG Value', fontsize=11)

ax.set_title('xG Shot Map — La Liga 2015/16\n(bubble size & colour = xG)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('xg_shot_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Player xG Leaderboard
Who overperformed or underperformed their xG?

In [ ]:
player_xg = (
    shots_df.groupby('player')
    .agg(
        shots=('goal', 'count'),
        goals=('goal', 'sum'),
        xg_sum=('xg', 'sum')
    )
    .query('shots >= 20')  # minimum shot filter
    .assign(xg_diff=lambda d: d['goals'] - d['xg_sum'])
    .sort_values('xg_diff', ascending=False)
    .reset_index()
)

print("=== Top 10 Overperformers (Goals − xG) ===")
print(player_xg.head(10)[['player','shots','goals','xg_sum','xg_diff']].to_string(index=False))

print("\n=== Top 10 Underperformers ===")
print(player_xg.tail(10)[['player','shots','goals','xg_sum','xg_diff']].to_string(index=False))

In [ ]:
# Visualise top 15 players by xG
top15 = player_xg.nlargest(15, 'xg_sum')

fig, ax = plt.subplots(figsize=(11, 6))
x_pos = np.arange(len(top15))
ax.bar(x_pos - 0.2, top15['goals'], width=0.4, label='Actual Goals', color='crimson', alpha=0.85)
ax.bar(x_pos + 0.2, top15['xg_sum'], width=0.4, label='xG (model)', color='steelblue', alpha=0.85)

ax.set_xticks(x_pos)
ax.set_xticklabels(top15['player'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Goals / xG')
ax.set_title('Actual Goals vs xG — Top 15 Players (La Liga 2015/16)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('player_goals_vs_xg.png', dpi=150)
plt.show()

## Summary

| Metric | Value |
|--------|-------|
| Model | Logistic Regression |
| Data | StatsBomb La Liga 2015/16 |
| Features | distance, angle, is_header, is_open_play |
| ROC-AUC | *see output above* |
| Brier Score | *see output above* |

### Possible Next Steps
- Add **shot pressure** feature (StatsBomb freeze frame data)
- Try **XGBoost / Random Forest** for non-linear relationships
- Extend to **multiple seasons / competitions**
- Build **interactive Plotly shot map**